In [ ]:
# Notebook 4/9 - Controls: Generation and Download
#
# Purpose: Generate a matched control article for each retracted (test) article and download
# the corresponding control PDF to Google Drive.
#
# High-level workflow:
# 1) Identify the set of retracted cases from OA PDFs stored in the Dissertation folder.
# 2) Build (overwrite) a cases database by joining case identifiers to metadata (DOI, title, year, journal, etc.).
# 3) For each case, search candidate controls using CORE/OpenAlex-based queries and structured matching criteria.
# 4) Apply a tiered matching ladder (strict to more permissive) with a hard per-case time budget.
# 5) Download the selected control PDF (safe-write + PDF signature validation).
# 6) Append a row to a persistent log after each attempt (resume-safe).
#
# Matching strategy (tiered control selection):
# - Control articles are selected using a hierarchical matching ladder that progressively relaxes constraints
#   if no suitable control is found at stricter levels.
# - The ladder is applied in order; the first successful match terminates the search for the current case.
# - A hard per-case time budget is enforced to ensure tractable execution and reproducibility.
#
# Matching levels (as implemented in this notebook):
# L1  : Same journal + exact publication year + subject + country + article type.
# L1b : Same journal + publication year within ±1 + subject + country + article type.
# L1c : Same journal + publication year within ±2 + subject + country + article type.
# L2  : Same journal + subject + country + article type (no year constraint).
# L3  : Any OA control from the same journal (criteria relaxed).
# L4  : Any OA control (final fallback; criteria relaxed).
#
# - This hierarchy prioritizes controls that are maximally comparable to the retracted article while still
#   ensuring a control can be identified when strict matches are unavailable.
# - The final match level achieved for each case is recorded in the control download log (match_level)
#   and is used later for audit and sensitivity analysis (including L4-specific diagnostics where applicable).
#
# Inputs (Google Drive):
# - cases_metadata.csv (case-level metadata used for matching)
# - oa_pdfs_all/ (folder containing retracted article PDFs; source of truth for which cases exist)
#
# Outputs (Google Drive):
# - retracted_cases_db.csv (overwritten each run; reflects the current set of cases in oa_pdfs_all/)
# - control_oa_pdfs_all/ (downloaded control PDFs)
# - controls_db.csv (metadata for downloaded controls)
# - control_download_log_all_new.csv (append-only attempt log)
#
# Execution:
# - Run the notebook top-to-bottom.
# - Reruns are resume-safe: existing valid control PDFs and logged attempts are skipped.
#
# Notes:
# This notebook is part of a larger, multi-stage data acquisition and
# analysis pipeline and is designed to be fully reproducible.

In [ ]:
# 1) Mount Google Drive + Install dependencies

from google.colab import drive
drive.mount("/content/drive")

!pip -q install pandas requests tqdm tenacity openpyxl

In [ ]:
# 2) Imports

import os, re, time, random, requests
import pandas as pd
import numpy as np
from tqdm import tqdm
from getpass import getpass
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

In [ ]:
# 3) Configuration

DISSERTATION_DIR = "/content/drive/MyDrive/Dissertation"

# INPUTS
METADATA_CSV = os.path.join(DISSERTATION_DIR, "cases_metadata.csv")

# Cases folder
OA_CASES_DIR = os.path.join(DISSERTATION_DIR, "oa_pdfs_all")

if not os.path.isdir(OA_CASES_DIR):
    raise FileNotFoundError(
        f"Required cases folder not found: {OA_CASES_DIR}"
    )

# OUTPUTS
CASES_DB_OUT = os.path.join(DISSERTATION_DIR, "retracted_cases_db.csv")

# Brand new controls folder
CONTROL_DIR = os.path.join(DISSERTATION_DIR, "control_oa_pdfs_all")
os.makedirs(CONTROL_DIR, exist_ok=True)

CONTROL_DB_OUT = os.path.join(DISSERTATION_DIR, "controls_db.csv")

CONTROL_LOG = os.path.join(DISSERTATION_DIR, "control_download_log_all_new.csv")

print("Using OA_CASES_DIR:", OA_CASES_DIR)
print("METADATA_CSV:", METADATA_CSV)
print("CASES_DB_OUT:", CASES_DB_OUT)
print("CONTROL_DIR:", CONTROL_DIR)
print("CONTROL_DB_OUT:", CONTROL_DB_OUT)
print("CONTROL_LOG:", CONTROL_LOG)

In [ ]:
# 4) CORE API Key (secure entry)

CORE_API_KEY = getpass("Paste CORE API key (hidden): ").strip()
if not CORE_API_KEY:
    raise ValueError("CORE API key is empty.")

CORE_API_BASE = "https://api.core.ac.uk/v3"
print("CORE key loaded for this session.")

In [ ]:
# 5) Helper functions

MIN_VALID_FILESIZE = 10_000
SLEEP_S = 0.25

def normalize_doi(d):
    if d is None or (isinstance(d, float) and pd.isna(d)):
        return None
    s = str(d).strip()
    if not s:
        return None
    s = s.replace("https://doi.org/", "").replace("http://doi.org/", "")
    return s.strip().lower()

def is_good_pdf(path):
    if not os.path.exists(path):
        return False
    if os.path.getsize(path) < MIN_VALID_FILESIZE:
        return False
    try:
        with open(path, "rb") as f:
            return f.read(5) == b"%PDF-"
    except Exception:
        return False

@retry(reraise=True, stop=stop_after_attempt(5),
       wait=wait_exponential(multiplier=1, min=1, max=20),
       retry=retry_if_exception_type(requests.RequestException))
def http_get(url, headers=None, params=None, timeout=60, stream=False):
    return requests.get(url, headers=headers or {}, params=params, timeout=timeout,
                        stream=stream, allow_redirects=True)

def download_pdf(url, out_path):
    """
    Downloads a PDF to out_path safely (writes .part then atomic rename).
    Validates PDF header.
    """
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/pdf,application/octet-stream;q=0.9,*/*;q=0.8",
    }
    tmp = out_path + ".part"
    r = http_get(url, headers=headers, timeout=90, stream=True)
    r.raise_for_status()

    with open(tmp, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 256):
            if chunk:
                f.write(chunk)

    with open(tmp, "rb") as f:
        if f.read(5) != b"%PDF-":
            try:
                os.remove(tmp)
            except Exception:
                pass
            return False, "Not a PDF"

    os.replace(tmp, out_path)
    return True, "OK"

def list_pdf_stems(folder):
    return sorted([
        os.path.splitext(f)[0].strip()
        for f in os.listdir(folder)
        if f.lower().endswith(".pdf")
    ])

def safe_str(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) else str(x).strip()

def norm_text(x):
    x = safe_str(x).lower()
    x = re.sub(r"\s+", " ", x).strip()
    return x

def year_to_int(y):
    y = safe_str(y)
    m = re.search(r"(19|20)\d{2}", y)
    return int(m.group(0)) if m else None

In [ ]:
# 6) Build Cases (always from scratch)

def norm_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = re.sub(r"\s+", " ", x)
    return x

def list_pdf_stems(folder):
    return sorted([
        os.path.splitext(f)[0]
        for f in os.listdir(folder)
        if f.lower().endswith(".pdf")
    ])

def extract_year(date_str):
    """Extract 4-digit year from strings like '2016-05-01' or '01/05/2016'."""
    if pd.isna(date_str):
        return ""
    s = str(date_str)
    m = re.search(r"(19\d{2}|20\d{2})", s)
    return m.group(1) if m else ""

# ---- STEP 1: enumerate ALL PDFs currently in folder ----
pdf_codes = list_pdf_stems(OA_CASES_DIR)

cases_db = pd.DataFrame({
    "code": pd.Series(pdf_codes, dtype="string")
})
cases_db["code"] = cases_db["code"].astype("string").str.strip()

print("PDFs detected in OA_CASES_DIR:", len(cases_db))
print("Example case codes:", cases_db["code"].head(10).tolist())

# ---- STEP 2: load Retraction Watch metadata ----
RW_PATH = os.path.join(DISSERTATION_DIR, "retraction_watch_database.xlsx")
rw = pd.read_excel(RW_PATH, dtype=str)
rw.columns = [c.strip() for c in rw.columns]

print("RW columns:", rw.columns.tolist())

# ---- STEP 3: mapping ----
# (Only require what exists; create blanks for non-existing fields.)
COLMAP_RW = {
    "code": "Record ID",
    "title": "Title",
    "subject": "Subject",
    "journal": "Journal",
    "publisher": "Publisher",
    "publication_date": "OriginalPaperDate",
    "doi": "OriginalPaperDOI",
    "article_type": "ArticleType",
    "authors": "Author",          # RW uses singular column name
    "institution": "Institution", # closest to affiliations
    "country": "Country",         # closest to author country signal you have
}

missing_required = [v for v in ["Record ID", "OriginalPaperDOI"] if v not in rw.columns]
if missing_required:
    raise ValueError(
        f"Missing REQUIRED columns in retraction_watch_database.xlsx: {missing_required}\n"
        f"Available columns: {rw.columns.tolist()}"
    )

# Normalize join key
rw["code"] = rw[COLMAP_RW["code"]].astype(str).str.strip()

# Select only columns that exist
rw_sel = pd.DataFrame()
rw_sel["code"] = rw["code"]

for out_col, src_col in COLMAP_RW.items():
    if out_col == "code":
        continue
    if src_col in rw.columns:
        rw_sel[out_col] = rw[src_col]
    else:
        rw_sel[out_col] = ""  # create empty column if missing

# Compute publication_year from publication_date
rw_sel["publication_year"] = rw_sel["publication_date"].apply(extract_year)

# ---- STEP 4: LEFT JOIN (folder defines the universe) ----
cases_db = cases_db.merge(rw_sel, on="code", how="left")

# ---- STEP 5: normalize matching variables ----
cases_db["doi"] = cases_db["doi"].astype(str).str.strip()
cases_db["year"] = cases_db["publication_year"].astype(str).str.strip()

cases_db["journal_norm"] = cases_db["journal"].apply(norm_text)
cases_db["publisher_norm"] = cases_db["publisher"].apply(norm_text)
cases_db["subject_norm"] = cases_db["subject"].apply(norm_text)
cases_db["article_type_norm"] = cases_db["article_type"].apply(norm_text)

# ---- STEP 6: diagnostics ----
cases_db["has_doi"] = cases_db["doi"].notna() & (cases_db["doi"] != "") & (cases_db["doi"] != "nan")
cases_db["has_subject"] = cases_db["subject_norm"] != ""
cases_db["has_year"] = cases_db["year"] != ""

print("\n--- CASES DB COVERAGE ---")
print("Total cases (PDFs):", len(cases_db))
print("Cases with DOI:", int(cases_db["has_doi"].sum()))
print("Cases with subject:", int(cases_db["has_subject"].sum()))
print("Cases with year:", int(cases_db["has_year"].sum()))

# Show rows that still failed to join (if any)
no_join = cases_db[~cases_db["has_doi"]].head(10)
if len(no_join) > 0:
    print("\nExample cases with missing DOI after join (first 10):")
    print(no_join[["code","doi","publication_date","journal","publisher"]].to_string(index=False))

cases_db.head(5)

In [ ]:
# 7) Export Cases (force overwrite)

# Safety: ensure cases_db exists
if "cases_db" not in globals():
    raise RuntimeError("cases_db not defined. Run CELL 4 first.")

cases_db["pdf_path"] = cases_db["code"].astype(str).str.strip().apply(
    lambda c: os.path.join(OA_CASES_DIR, f"{c}.pdf")
)

# Force overwrite
cases_db.to_csv(CASES_DB_OUT, index=False, encoding="utf-8-sig")

print("CASES_DB_OUT overwritten:", CASES_DB_OUT)
print("Rows written:", len(cases_db))
print("PDFs in folder:", len([f for f in os.listdir(OA_CASES_DIR) if f.lower().endswith('.pdf')]))

In [ ]:
# 8) Prepare control log + identify cases missing controls

LOG_COLS = [
    "case_code","case_doi","status","match_level","subject_used",
    "control_doi","control_title","control_journal","control_publisher","control_year","control_type",
    "source","pdf_url","filepath","message"
]

if not os.path.exists(CONTROL_LOG):
    pd.DataFrame(columns=LOG_COLS).to_csv(CONTROL_LOG, index=False)

control_log = pd.read_csv(CONTROL_LOG)

# Never select these as controls
retracted_dois = set([d for d in cases_db["doi"].astype(str).tolist() if d and d != "nan"])

# Track used control DOIs (avoid duplicates across cases)
used_control_dois = set(
    control_log.loc[control_log["status"]=="downloaded", "control_doi"]
      .astype(str).str.strip().tolist()
)
used_control_dois = set([d for d in used_control_dois if d and d != "nan"])

def has_valid_control(case_code):
    p = os.path.join(CONTROL_DIR, f"C{case_code}.pdf")
    return is_good_pdf(p)

cases_db["has_control"] = cases_db["code"].apply(has_valid_control)

print("Total cases:", len(cases_db))
print("Cases with control already:", int(cases_db["has_control"].sum()))
print("Cases missing control:", int((~cases_db["has_control"]).sum()))

# Only process cases that still do not have a control file on disk
to_process = cases_db[~cases_db["has_control"]].copy()
print("Cases to process this run:", len(to_process))

# Avoid re-processing cases already logged as downloaded
downloaded_cases_in_log = set(
    control_log.loc[control_log["status"]=="downloaded", "case_code"]
      .astype(str).str.strip().tolist()
)
to_process = to_process[~to_process["code"].isin(downloaded_cases_in_log)].copy()
print("Cases to process after excluding log-downloaded:", len(to_process))

In [ ]:
# 9) CORE API — robust client with 429 backoff + safe parsing

CORE_API_BASE = "https://api.core.ac.uk/v3"

def core_headers():
    return {"Authorization": f"Bearer {CORE_API_KEY}"}

def core_headers_xkey():
    return {"X-CORE-API-Key": CORE_API_KEY}

def looks_like_pdf(u):
    if not u:
        return False
    u2 = u.lower()
    return (".pdf" in u2) or u2.endswith("/pdf") or u2.endswith("/pdf/")

def extract_urls_from_core_record(rec):
    urls = []
    for k in ("downloadUrl","fullTextUrl","pdfUrl","url"):
        v = rec.get(k)
        if isinstance(v, str) and v:
            urls.append(v)

    for k in ("links","fullText","fullTextLinks","repositories"):
        v = rec.get(k)
        if isinstance(v, list):
            for it in v:
                if isinstance(it, dict):
                    for kk in ("url","downloadUrl","link","pdfUrl","fullTextUrl"):
                        vv = it.get(kk)
                        if isinstance(vv, str) and vv:
                            urls.append(vv)
        elif isinstance(v, dict):
            for kk in ("url","downloadUrl","pdf","link","pdfUrl","fullTextUrl"):
                vv = v.get(kk)
                if isinstance(vv, str) and vv:
                    urls.append(vv)

    seen=set(); out=[]
    for u in urls:
        if u not in seen:
            seen.add(u); out.append(u)
    return out

def parse_core_record(rec):
    title = rec.get("title") if isinstance(rec.get("title"), str) else ""
    doi = rec.get("doi") if isinstance(rec.get("doi"), str) else ""
    year = rec.get("yearPublished") or rec.get("publishedYear") or rec.get("year")
    year = year_to_int(year)

    journal = rec.get("journal") or rec.get("journalTitle") or ""
    publisher = rec.get("publisher") or ""
    doc_type = rec.get("documentType") or rec.get("type") or ""

    return {
        "title": safe_str(title),
        "doi": normalize_doi(doi),
        "year": year,
        "journal": safe_str(journal),
        "publisher": safe_str(publisher),
        "type": safe_str(doc_type),
        "urls": extract_urls_from_core_record(rec),
    }

def core_request_json(url, params=None, timeout=30, max_retries=8, base_sleep=1.0):
    """
    Robust CORE request with retry/backoff on 429/5xx.
    Tries both auth styles.
    """
    headers_list = [core_headers(), core_headers_xkey()]
    auth_labels = ["Authorization: Bearer", "X-CORE-API-Key"]

    last_exc = None

    for attempt in range(max_retries):
        for headers, label in zip(headers_list, auth_labels):
            try:
                r = requests.get(url, params=params, headers=headers, timeout=timeout)

                # Rate-limited
                if r.status_code == 429:
                    retry_after = r.headers.get("Retry-After")
                    sleep_s = float(retry_after) if retry_after and retry_after.isdigit() else (base_sleep * (2 ** attempt))
                    sleep_s = min(sleep_s, 120.0)
                    time.sleep(sleep_s)
                    continue

                # Temporary server issues
                if r.status_code in (500, 502, 503, 504):
                    time.sleep(min(base_sleep * (2 ** attempt), 60.0))
                    continue

                # Auth errors: try the other header
                if r.status_code in (401, 403):
                    continue

                r.raise_for_status()
                return r.json(), label

            except requests.RequestException as e:
                last_exc = e
                time.sleep(min(base_sleep * (2 ** attempt), 60.0))
                continue

    raise last_exc if last_exc else RuntimeError("CORE request failed without exception")

def core_search_works(q, limit=25, offset=0):
    url = f"{CORE_API_BASE}/search/works"
    js, auth_used = core_request_json(url, params={"q": q, "limit": limit, "offset": offset}, timeout=30)
    results = js.get("results", []) if isinstance(js, dict) else []
    return results, auth_used

In [ ]:
# 10) Control matching + download

# Ensure LOG_COLS exists; if not, define it here too
LOG_COLS = [
    "case_code","case_doi","status","match_level","subject_used",
    "control_doi","control_title","control_journal","control_publisher","control_year","control_type",
    "source","pdf_url","filepath","message"
]

def build_queries_for_case(journal, publisher, year, art_type, subject):
    if not subject:
        return []

    journal_q = f'"{journal}"' if journal else ""
    publisher_q = f'"{publisher}"' if publisher else ""
    type_q = f'"{art_type}"' if art_type else ""
    subj_q = f'"{subject}"'

    queries = []
    if journal and art_type and year:
        queries.append(("L1a", f"{journal_q} AND {type_q} AND {subj_q} AND year:{int(year)}", 0))
        queries.append(("L1b", f"{journal_q} AND {type_q} AND {subj_q}", 1))
        queries.append(("L1c", f"{journal_q} AND {type_q} AND {subj_q}", 2))
    if journal and art_type:
        queries.append(("L2", f"{journal_q} AND {type_q} AND {subj_q}", None))
    if publisher and art_type:
        queries.append(("L3", f"{publisher_q} AND {type_q} AND {subj_q}", None))
    return queries

def within_year_window(candidate_year, target_year, window):
    if window is None:
        return True
    if candidate_year is None or target_year is None:
        return False
    return abs(int(candidate_year) - int(target_year)) <= int(window)

def log_row(**kwargs):
    """
    Always returns a dict with exactly LOG_COLS keys.
    Missing keys filled with "".
    """
    row = {k: "" for k in LOG_COLS}
    for k, v in kwargs.items():
        if k in row:
            row[k] = v
    return row

def choose_and_download_control_for_case(case_row, max_pages_per_level=4, page_size=25):
    """
    Downloads ONE control into CONTROL_DIR as C<case_code>.pdf
    Returns dict row if downloaded; otherwise None (but logs failure).
    """
    def get_col(row, *names):
        for n in names:
            if n in row and pd.notna(row[n]):
                return row[n]
        return ""

    case_code = str(case_row["code"]).strip()
    case_doi  = normalize_doi(case_row.get("doi", ""))

    journal   = norm_text(get_col(case_row, "journal", "journal_norm"))
    publisher = norm_text(get_col(case_row, "publisher", "publisher_norm"))
    subject   = norm_text(get_col(case_row, "subject", "subject_norm"))
    art_type  = norm_text(get_col(case_row, "article_type", "article_type_norm"))
    year      = case_row.get("year", None)

    out_pdf = os.path.join(CONTROL_DIR, f"C{case_code}.pdf")

    # Disk-first skip
    if is_good_pdf(out_pdf):
        row = log_row(
            case_code=case_code, case_doi=case_doi, status="downloaded",
            match_level="existing_file", subject_used=subject,
            source="disk", filepath=out_pdf, message="File already existed"
        )
        control_log.loc[len(control_log)] = [row[c] for c in LOG_COLS]
        return row

    queries = build_queries_for_case(journal, publisher, year, art_type, subject)
    if not queries:
        row = log_row(
            case_code=case_code, case_doi=case_doi, status="failed",
            match_level="no_query", subject_used=subject,
            source="core", filepath=out_pdf,
            message="Insufficient metadata (subject mandatory; journal/type/publisher missing)."
        )
        control_log.loc[len(control_log)] = [row[c] for c in LOG_COLS]
        return None

    for level_name, q, year_window in queries:
        for page in range(max_pages_per_level):
            offset = page * page_size

            try:
                results, auth_used = core_search_works(q=q, limit=page_size, offset=offset)
            except Exception as e:
                row = log_row(
                    case_code=case_code, case_doi=case_doi, status="failed",
                    match_level=level_name, subject_used=subject,
                    source="core", filepath=out_pdf,
                    message=f"CORE error: {str(e)}"
                )
                control_log.loc[len(control_log)] = [row[c] for c in LOG_COLS]
                return None

            if not results:
                break

            for rec in results:
                parsed = parse_core_record(rec)
                cdoi = parsed["doi"]

                if not cdoi:
                    continue
                if cdoi in retracted_dois:
                    continue
                if cdoi in used_control_dois:
                    continue
                if not within_year_window(parsed.get("year"), year, year_window):
                    continue

                urls = parsed["urls"]
                if not urls:
                    continue

                pdf_first = [u for u in urls if looks_like_pdf(u)]
                others = [u for u in urls if u not in pdf_first]
                ordered = pdf_first + others

                last_msg, last_url = "", ""
                for url in ordered[:8]:
                    try:
                        ok, msg = download_pdf(url, out_pdf)
                        if ok and is_good_pdf(out_pdf):
                            used_control_dois.add(cdoi)
                            row = log_row(
                                case_code=case_code, case_doi=case_doi, status="downloaded",
                                match_level=level_name, subject_used=subject,
                                control_doi=cdoi,
                                control_title=parsed.get("title",""),
                                control_journal=parsed.get("journal",""),
                                control_publisher=parsed.get("publisher",""),
                                control_year=parsed.get("year",""),
                                control_type=parsed.get("type",""),
                                source=f"core ({auth_used})",
                                pdf_url=url, filepath=out_pdf, message="OK"
                            )
                            control_log.loc[len(control_log)] = [row[c] for c in LOG_COLS]
                            return row
                        else:
                            last_msg, last_url = msg, url
                    except Exception as e:
                        last_msg, last_url = str(e), url

                    time.sleep(SLEEP_S)

            time.sleep(SLEEP_S)

    row = log_row(
        case_code=case_code, case_doi=case_doi, status="failed",
        match_level="L1a→L1b→L1c→L2→L3", subject_used=subject,
        source="core", filepath=out_pdf,
        message="No suitable OA control found/downloaded within search limits."
    )
    control_log.loc[len(control_log)] = [row[c] for c in LOG_COLS]
    return None

In [ ]:
# 11) Fast Control Download

# 11.1) Configuration
OPENALEX = "https://api.openalex.org"
UNPAYWALL = "https://api.unpaywall.org/v2"

USE_CORE = True          # requires core_search_works() from earlier cell (optional)
USE_UNPAYWALL = True     # set False if you do NOT want Unpaywall
UNPAYWALL_EMAIL = "xxxxxxxxxxxxx"
OPENALEX_MAILTO = UNPAYWALL_EMAIL

# HARD per-case budget (TOTAL, includes everything: search + download + backoff)
TOTAL_CASE_BUDGET_S = 5 * 60   # 5 minutes TOTAL per case

# Keep search light to avoid runaway loops
PAGE_SIZE = 10
MAX_PAGES_PER_LEVEL = 6

# Per-request timeouts
OPENALEX_TIMEOUT_S = 12
UNPAYWALL_TIMEOUT_S = 12
PDF_CONNECT_TIMEOUT_S = 10
PDF_READ_TIMEOUT_S = 18

# Rate-limit/ sleep behavior
BASE_SLEEP_S = 0.10
MAX_BACKOFF_SLEEP_S = 2.0

MIN_VALID_BYTES = 10_000
ENFORCE_UNIQUE_CONTROL_DOI = True

# Path to RW metadata (must exist)
RW_META_XLSX = "/content/drive/MyDrive/Dissertation/retraction_watch_database.xlsx"
if not os.path.exists(RW_META_XLSX):
    raise FileNotFoundError(f"Retraction Watch file not found at: {RW_META_XLSX}")

print(
    f"⚙️ Per-case hard budget: {TOTAL_CASE_BUDGET_S}s "
    f"(OpenAlex timeout={OPENALEX_TIMEOUT_S}s, PDF timeouts={PDF_CONNECT_TIMEOUT_S}/{PDF_READ_TIMEOUT_S}s)"
)

# 11.2) Load Cases DB
cases = pd.read_csv(CASES_DB_OUT, encoding="utf-8-sig", dtype=str)
cases.columns = [c.strip() for c in cases.columns]
cases["code"] = cases["code"].astype(str).str.strip()

if "publisher'subject" in cases.columns and "subject" not in cases.columns:
    raise ValueError(
        "Your cases DB has a corrupted column name: 'publisher'subject'. "
        "Please re-create CASES_DB_OUT ensuring commas are correct in CSV writing."
    )

required = ["code","doi","year","journal","publisher","subject","article_type",
            "journal_norm","publisher_norm","subject_norm","article_type_norm"]
missing = [c for c in required if c not in cases.columns]
if missing:
    raise ValueError(f"CASES_DB_OUT missing columns: {missing}. Found: {cases.columns.tolist()}")

def norm_doi(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    s = str(x).strip().lower()
    s = s.replace("https://doi.org/","").replace("http://doi.org/","")
    return s

retracted_dois = set(cases["doi"].dropna().astype(str).map(norm_doi).tolist())

# 11.3) Load RW metadata for case country
rw = pd.read_excel(RW_META_XLSX, dtype=str)
rw.columns = [c.strip() for c in rw.columns]

if "Record ID" not in rw.columns or "Country" not in rw.columns:
    raise ValueError(f"RW file must include 'Record ID' and 'Country'. Found: {rw.columns.tolist()}")

rw["Record ID"] = rw["Record ID"].astype(str).str.strip()
rw["Country"] = rw["Country"].astype(str).str.strip()

rw_country_by_code = dict(zip(rw["Record ID"], rw["Country"]))
cases["country_raw"] = cases["code"].map(rw_country_by_code).fillna("")

COUNTRY_NAME_TO_ISO2 = {
    "united states": "US", "usa": "US", "u.s.a.": "US", "us": "US", "u.s.": "US", "america": "US",
    "united kingdom": "GB", "uk": "GB", "u.k.": "GB", "great britain": "GB", "britain": "GB",
    "england": "GB", "scotland": "GB", "wales": "GB", "northern ireland": "GB",
    "ireland": "IE", "republic of ireland": "IE",
    "portugal": "PT", "spain": "ES", "france": "FR", "germany": "DE", "italy": "IT",
    "netherlands": "NL", "belgium": "BE", "switzerland": "CH", "austria": "AT",
    "sweden": "SE", "norway": "NO", "denmark": "DK", "finland": "FI",
    "poland": "PL", "greece": "GR", "china": "CN", "japan": "JP", "india": "IN",
    "canada": "CA", "australia": "AU", "new zealand": "NZ", "brazil": "BR",
}
def country_name_to_iso2(name: str) -> str:
    if not name:
        return ""
    s = str(name).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return COUNTRY_NAME_TO_ISO2.get(s, "")

cases["country_iso2"] = cases["country_raw"].apply(country_name_to_iso2)

# 11.4) ArticleType mapping
def norm_text(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    return re.sub(r"\s+", " ", str(x).strip().lower())

ARTICLETYPE_TO_OPENALEX = {
    "journal article": "journal-article",
    "journal-article": "journal-article",
    "article": "journal-article",
    "research article": "journal-article",
    "review": "review",
    "review article": "review",
    "proceedings article": "proceedings-article",
    "conference paper": "proceedings-article",
    "conference article": "proceedings-article",
    "proceedings-article": "proceedings-article",
    "book chapter": "book-chapter",
    "book-chapter": "book-chapter",
    "preprint": "preprint",
    "dataset": "dataset",
    "editorial": "editorial",
    "letter": "letter",
    "retraction": "retraction",
    "correction": "correction",
    "erratum": "correction",
}
CANON_OPENALEX_TYPES = {
    "journal-article","proceedings-article","book-chapter","preprint","review",
    "dataset","editorial","letter","retraction","correction"
}

def case_article_type_target(case_row) -> str:
    raw = case_row.get("article_type_norm", "") or case_row.get("article_type", "")
    s = norm_text(raw)
    if not s:
        return ""
    return ARTICLETYPE_TO_OPENALEX.get(s, s)

def openalex_work_type(work) -> str:
    return norm_text(work.get("type", ""))

def article_type_matches(case_target: str, work_type: str) -> bool:
    case_target = norm_text(case_target)
    work_type = norm_text(work_type)
    if not case_target:
        return True
    if not work_type:
        return False
    if case_target in CANON_OPENALEX_TYPES:
        return work_type == case_target
    return (case_target in work_type) or (work_type in case_target)

# 11.5) Control folder + validity checks
os.makedirs(CONTROL_DIR, exist_ok=True)

def control_path(code):
    return os.path.join(CONTROL_DIR, f"C{str(code).strip()}.pdf")

def is_good_pdf_local(path):
    try:
        if not os.path.exists(path): return False
        if os.path.getsize(path) < MIN_VALID_BYTES: return False
        with open(path, "rb") as f:
            return f.read(5) == b"%PDF-"
    except Exception:
        return False

def already_has_control(code):
    return is_good_pdf_local(control_path(code))

# 11.6) Log setup
LOG_COLS = [
    "case_code","case_doi","status","match_level","dropped_constraints",
    "control_doi","control_title","control_journal","control_publisher","control_year","control_type","control_subject",
    "source","pdf_url","filepath","message",
    "case_country_iso2","control_first_author_country_iso2"
]

if os.path.exists(CONTROL_LOG):
    control_log = pd.read_csv(CONTROL_LOG, dtype=str)
    if list(control_log.columns) != LOG_COLS:
        print("CONTROL_LOG schema mismatch. Resetting:", CONTROL_LOG)
        control_log = pd.DataFrame(columns=LOG_COLS)
        control_log.to_csv(CONTROL_LOG, index=False)
else:
    control_log = pd.DataFrame(columns=LOG_COLS)
    control_log.to_csv(CONTROL_LOG, index=False)

used_control_dois = set(
    control_log.loc[control_log["status"]=="downloaded","control_doi"]
    .dropna().astype(str).map(norm_doi).tolist()
)

# 11.7) Deadline-aware helpers
def remaining_s(deadline):
    return max(0.0, deadline - time.monotonic())

def sleep_bounded(seconds, deadline):
    rs = remaining_s(deadline)
    if rs <= 0:
        return
    time.sleep(min(seconds, rs))

def openalex_get(url, params=None, deadline=None):
    params = params or {}
    if OPENALEX_MAILTO and "mailto" not in params:
        params["mailto"] = OPENALEX_MAILTO
    to = min(OPENALEX_TIMEOUT_S, max(1, int(remaining_s(deadline))) ) if deadline else OPENALEX_TIMEOUT_S
    r = requests.get(url, params=params, timeout=to)
    r.raise_for_status()
    return r.json()

def backoff_sleep(attempt, deadline):
    # Small sleeps only; never exceed deadline
    s = min(MAX_BACKOFF_SLEEP_S, (2 ** attempt) * 0.25 + random.random() * 0.25)
    sleep_bounded(s, deadline)

def download_pdf_strict(url, out_path, deadline):
    """
    Strict downloader that cannot exceed remaining deadline.
    """
    rs = remaining_s(deadline)
    if rs <= 1:
        return False, "No time remaining for download"

    # Use remaining budget to bound timeouts
    # Keep connect/ read separate, both capped by remaining time
    connect_to = min(PDF_CONNECT_TIMEOUT_S, max(1, int(rs)))
    read_to = min(PDF_READ_TIMEOUT_S, max(1, int(rs)))

    tmp = out_path + ".part"
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/pdf,application/octet-stream;q=0.9,*/*;q=0.8",
    }
    try:
        r = requests.get(
            url,
            headers=headers,
            timeout=(connect_to, read_to),
            stream=True,
            allow_redirects=True
        )
        if r.status_code in (403,404,410):
            return False, f"HTTP {r.status_code}"
        r.raise_for_status()

        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*256):
                if chunk:
                    f.write(chunk)
                if remaining_s(deadline) <= 0:
                    try:
                        if os.path.exists(tmp): os.remove(tmp)
                    except:
                        pass
                    return False, "Deadline exceeded during download"

        with open(tmp, "rb") as f:
            if f.read(5) != b"%PDF-":
                os.remove(tmp)
                return False, "Not a PDF"

        os.replace(tmp, out_path)
        return True, "OK"
    except Exception as e:
        try:
            if os.path.exists(tmp): os.remove(tmp)
        except:
            pass
        return False, str(e)

# 11.8) OpenAlex search helpers
def openalex_find(source_id=None, year=None, search_kw=None, cursor="*", per_page=PAGE_SIZE, deadline=None):
    filt = ["open_access.is_oa:true", "has_doi:true"]
    if source_id:
        filt.append(f"primary_location.source.id:{source_id}")
    if year and str(year).isdigit():
        filt.append(f"publication_year:{int(year)}")

    params = {"filter": ",".join(filt), "per-page": per_page, "cursor": cursor}
    if search_kw:
        params["search"] = search_kw

    data = openalex_get(f"{OPENALEX}/works", params=params, deadline=deadline)
    return data.get("results", []), data.get("meta", {}).get("next_cursor", None)

def oa_doi_from_openalex(work):
    doi = (work.get("doi") or "")
    doi = doi.replace("https://doi.org/","").replace("http://doi.org/","")
    return norm_doi(doi)

def openalex_pdf_url(work):
    pl = work.get("primary_location") or {}
    if pl.get("pdf_url"):
        return pl["pdf_url"]
    for loc in (work.get("locations") or []):
        if loc.get("pdf_url"):
            return loc["pdf_url"]
    return ""

def openalex_first_author_country_iso2(work) -> str:
    authorships = work.get("authorships") or []
    if not authorships:
        return ""
    def country_from_auth(auth):
        for inst in (auth.get("institutions") or []):
            cc = inst.get("country_code")
            if cc:
                return str(cc).upper()
        return ""
    for a in authorships:
        if str(a.get("author_position","")).lower() == "first":
            cc = country_from_auth(a)
            if cc: return cc
            break
    cc = country_from_auth(authorships[0])
    if cc: return cc
    for a in authorships:
        if a.get("is_corresponding"):
            cc = country_from_auth(a)
            if cc: return cc
    return ""

# 11.9) Source ID resolver (journal -> OpenAlex source)
source_cache = {}
def resolve_source_id(journal_name, deadline=None):
    j = (journal_name or "").strip()
    if not j:
        return ""
    key = j.lower()
    if key in source_cache:
        return source_cache[key]
    data = openalex_get(f"{OPENALEX}/sources", params={"search": j, "per-page": 5}, deadline=deadline)
    res = data.get("results", []) or []
    sid = res[0]["id"] if res else ""
    source_cache[key] = sid
    return sid

# 11.10) CORE and Unpaywall (both deadline-aware)
def core_pdf_by_doi(doi, deadline):
    if not USE_CORE or "core_search_works" not in globals():
        return ""
    q = f"doi:{doi}"
    for attempt in range(5):
        if remaining_s(deadline) <= 0:
            return ""
        try:
            results, _ = core_search_works(q=q, limit=5, offset=0)
            for w in results or []:
                for k in ["downloadUrl","fullTextIdentifier","pdfUrl","url"]:
                    u = w.get(k)
                    if isinstance(u, str) and u.startswith("http"):
                        return u
            return ""
        except requests.HTTPError as e:
            code = e.response.status_code if e.response is not None else None
            if code == 429:
                backoff_sleep(attempt, deadline); continue
            return ""
        except Exception:
            backoff_sleep(attempt, deadline)
    return ""

def unpaywall_best_pdf(doi, deadline):
    if not USE_UNPAYWALL:
        return ""
    doi = norm_doi(doi)
    if not doi:
        return ""
    if not UNPAYWALL_EMAIL or "@" not in UNPAYWALL_EMAIL:
        raise ValueError("UNPAYWALL_EMAIL must be a valid email address.")

    for attempt in range(5):
        if remaining_s(deadline) <= 0:
            return ""
        try:
            to = min(UNPAYWALL_TIMEOUT_S, max(1, int(remaining_s(deadline))))
            r = requests.get(f"{UNPAYWALL}/{doi}", params={"email": UNPAYWALL_EMAIL}, timeout=to)
            if r.status_code == 429:
                backoff_sleep(attempt, deadline); continue
            if r.status_code == 404:
                return ""
            r.raise_for_status()
            data = r.json()
            bol = data.get("best_oa_location") or {}
            if bol.get("url_for_pdf"):
                return bol["url_for_pdf"]
            for loc in (data.get("oa_locations") or []):
                if loc.get("url_for_pdf"):
                    return loc["url_for_pdf"]
            return ""
        except Exception:
            backoff_sleep(attempt, deadline)
    return ""

def pdf_url_for_doi(doi_c, work, deadline):
    u = core_pdf_by_doi(doi_c, deadline)
    if not u:
        u = openalex_pdf_url(work)
    if not u:
        u = unpaywall_best_pdf(doi_c, deadline)
    return u

# 11.11) Subject keyword extraction
def subject_keyword(subject_str):
    if not subject_str or str(subject_str).strip().lower() in ("nan","none",""):
        return ""
    s = str(subject_str)
    parts = [p.strip() for p in s.split(";") if p.strip()]
    if not parts:
        return ""
    p0 = re.sub(r"^\([^)]*\)\s*", "", parts[0]).strip()
    p0 = re.sub(r"\s+", " ", p0).strip()
    return p0[:80]

# 11.12) Ladder levels
LADDER = [
    ("L1",  "exact", "same journal + year exact + subject + country + article_type", True,  True),
    ("L1b", "pm1",   "same journal + year ±1 + subject + country + article_type",   True,  True),
    ("L1c", "pm2",   "same journal + year ±2 + subject + country + article_type",   True,  True),
    ("L2",  "none",  "same journal + subject + country + article_type (no year)",   True,  True),
    ("L3",  "any",   "any OA from the same journal (criteria relaxed)",             False, False),
    ("L4",  "any",   "any OA (final fallback; criteria relaxed)",                   False, False),
]

def fail_result(case_code, case_doi, subj_kw, out_path, case_country_iso2, level, msg):
    return {
        "case_code": case_code,
        "case_doi": case_doi,
        "status": "failed",
        "match_level": level,
        "dropped_constraints": msg,
        "control_doi": "",
        "control_title": "",
        "control_journal": "",
        "control_publisher": "",
        "control_year": "",
        "control_type": "",
        "control_subject": subj_kw,
        "source": "openalex_match + core_pdf_first",
        "pdf_url": "",
        "filepath": out_path,
        "message": msg,
        "case_country_iso2": case_country_iso2,
        "control_first_author_country_iso2": ""
    }

# 11.13) Core resolver per case (deadline-enforced)
def find_and_download_control(case_row):
    start = time.monotonic()
    deadline = start + TOTAL_CASE_BUDGET_S

    case_code = str(case_row["code"]).strip()
    case_doi  = norm_doi(case_row.get("doi",""))
    year      = str(case_row.get("year","")).strip()
    journal   = str(case_row.get("journal","")).strip()
    publisher = str(case_row.get("publisher","")).strip()
    subject   = str(case_row.get("subject","")).strip()

    subj_kw = subject_keyword(subject)
    out_path = control_path(case_code)

    # case country from RW
    case_country_raw = ""
    if (cases["code"] == case_code).any():
        case_country_raw = cases.loc[cases["code"] == case_code, "country_raw"].values[0]
    case_country_iso2 = country_name_to_iso2(case_country_raw)
    has_case_country = bool(case_country_iso2)

    # case type target
    case_type_target = case_article_type_target(case_row)
    has_case_type = bool(case_type_target)

    # Resolve source_id with deadline
    try:
        source_id = resolve_source_id(journal, deadline=deadline)
    except Exception:
        source_id = ""

    # Helper: attempt candidates from a given OpenAlex query
    def try_query(level, source_id_q, year_q, search_kw_q, enforce_country, enforce_type):
        cursor = "*"
        pages = 0

        while cursor and pages < MAX_PAGES_PER_LEVEL and remaining_s(deadline) > 0:
            pages += 1
            try:
                results, cursor = openalex_find(
                    source_id=source_id_q,
                    year=year_q,
                    search_kw=search_kw_q,
                    cursor=cursor,
                    per_page=PAGE_SIZE,
                    deadline=deadline
                )
            except Exception:
                # If OpenAlex hiccups, don’t get stuck: move on quickly
                sleep_bounded(BASE_SLEEP_S, deadline)
                continue

            if not results:
                break

            random.shuffle(results)

            for w in results:
                if remaining_s(deadline) <= 0:
                    break

                doi_c = oa_doi_from_openalex(w)
                if not doi_c:
                    continue
                if doi_c == case_doi:
                    continue
                if doi_c in retracted_dois:
                    continue
                if ENFORCE_UNIQUE_CONTROL_DOI and doi_c in used_control_dois:
                    continue

                # Enforced criteria only where requested
                ctrl_first_cc = openalex_first_author_country_iso2(w)
                if enforce_country:
                    if not ctrl_first_cc or ctrl_first_cc != case_country_iso2:
                        continue

                ctrl_type = openalex_work_type(w)
                if enforce_type:
                    if not ctrl_type or not article_type_matches(case_type_target, ctrl_type):
                        continue

                pdf_url = pdf_url_for_doi(doi_c, w, deadline)
                if not pdf_url:
                    continue

                ok, msg = download_pdf_strict(pdf_url, out_path, deadline)
                if ok and is_good_pdf_local(out_path):
                    used_control_dois.add(doi_c)
                    return {
                        "case_code": case_code,
                        "case_doi": case_doi,
                        "status": "downloaded",
                        "match_level": level,
                        "dropped_constraints": dict((lv, d) for lv,_,d,_,_ in LADDER).get(level, ""),
                        "control_doi": doi_c,
                        "control_title": (w.get("title") or ""),
                        "control_journal": journal,
                        "control_publisher": publisher,
                        "control_year": str(w.get("publication_year") or ""),
                        "control_type": ctrl_type,
                        "control_subject": subj_kw,
                        "source": "openalex_match + core_pdf_first",
                        "pdf_url": pdf_url,
                        "filepath": out_path,
                        "message": msg,
                        "case_country_iso2": case_country_iso2,
                        "control_first_author_country_iso2": ctrl_first_cc
                    }

                # small bounded pause
                sleep_bounded(BASE_SLEEP_S, deadline)

            sleep_bounded(BASE_SLEEP_S, deadline)

        return None

    # Iterate ladder with deadline enforced
    for level, ymode, desc, require_country, require_type in LADDER:
        if remaining_s(deadline) <= 0:
            break

        # enforcement only in L1/L1b/L1c/L2
        enforce_country = require_country and has_case_country and level in {"L1","L1b","L1c","L2"}
        enforce_type = require_type and has_case_type and level in {"L1","L1b","L1c","L2"}

        # decide years
        years_to_try = [None]
        if level in {"L1","L1b","L1c","L2"}:
            if ymode == "exact" and year.isdigit():
                years_to_try = [int(year)]
            elif ymode == "pm1" and year.isdigit():
                y = int(year); years_to_try = [y, y-1, y+1]
            elif ymode == "pm2" and year.isdigit():
                y = int(year); years_to_try = [y, y-1, y+1, y-2, y+2]
            else:
                years_to_try = [None]

        # L4 ignores journal; L3 requires journal (source_id)
        for y in years_to_try:
            if remaining_s(deadline) <= 0:
                break

            if level == "L4":
                sid = None
                kw = subj_kw if subj_kw else "open access"
                res = try_query(level, sid, None, kw, False, False)
            elif level == "L3":
                if not source_id:
                    continue
                sid = source_id
                kw = subj_kw if subj_kw else "open access"
                res = try_query(level, sid, None, kw, False, False)
            else:
                # L1/L1b/L1c/L2: require subject keyword to be meaningful; if missing, skip directly to L3/L4
                if not subj_kw:
                    continue
                sid = source_id
                kw = subj_kw
                res = try_query(level, sid, y, kw, enforce_country, enforce_type)

            if res is not None:
                res["dropped_constraints"] = desc
                return res

    # If we get here, deadline hit without any downloadable OA PDF
    return fail_result(
        case_code, case_doi, subj_kw, out_path, case_country_iso2,
        level="TIMEOUT",
        msg=f"No OA PDF downloadable within {TOTAL_CASE_BUDGET_S}s hard budget."
    )

# 11.14) Run (only missing controls)
cases["has_control"] = cases["code"].apply(already_has_control)
pending = cases[~cases["has_control"]].copy()

print("Total cases (from disk):", len(cases))
print("Cases with control already:", int(cases["has_control"].sum()))
print("Cases still needing control:", len(pending))

pbar = tqdm(pending.itertuples(index=False), total=len(pending), desc="Download controls", unit="case")

for r in pbar:
    row = r._asdict()
    code = str(row["code"]).strip()

    if already_has_control(code):
        continue

    target = control_path(code)
    print(f"Case {code} → target: {target}")

    out = find_and_download_control(row)

    control_log.loc[len(control_log)] = [out.get(c, "") for c in LOG_COLS]

    # periodic save
    if len(control_log) % 50 == 0:
        control_log.to_csv(CONTROL_LOG, index=False)

    if out["status"] == "downloaded":
        print(f"({out['match_level']}) {out.get('control_doi','')} → {out.get('filepath','')}")
    else:
        print(f"{code} failed: {out.get('message','')}")

control_log.to_csv(CONTROL_LOG, index=False)

print("\n--- SUMMARY ---")
print("Downloaded controls:", int((control_log["status"]=="downloaded").sum()))
print("Failed cases:", int((control_log["status"]=="failed").sum()))
print("Control folder:", CONTROL_DIR)
print("Log:", CONTROL_LOG)

In [ ]:
# 12) Export CONTROL database CSV + sanity checks

# Reload log safely
control_log = pd.read_csv(CONTROL_LOG)

# Keep only successfully downloaded controls
controls_db = control_log[control_log["status"] == "downloaded"].copy()

# Save controls database
controls_db.to_csv(CONTROL_DB_OUT, index=False, encoding="utf-8-sig")

print("Saved controls DB:", CONTROL_DB_OUT)
print("Downloaded controls total (log):", len(controls_db))

# Sanity checks

# Check for more than one control per case (should normally be 1)
counts = controls_db.groupby("case_code").size()

if not counts.empty:
    print("Max controls for any case:", int(counts.max()))
    dup_cases = counts[counts > 1]
    print("Cases with >1 control:", len(dup_cases))
else:
    print("No downloaded controls logged yet.")

# Check how many cases are still missing a control (disk is source of truth)
cases_db2 = pd.read_csv(CASES_DB_OUT, encoding="utf-8-sig")

def has_control_file(code):
    return os.path.exists(os.path.join(CONTROL_DIR, f"C{code}.pdf"))

cases_db2["has_control_now"] = cases_db2["code"].astype(str).apply(has_control_file)

remaining = cases_db2[~cases_db2["has_control_now"]]

print("Remaining cases without control PDFs:", len(remaining))

controls_db.head(10)

In [ ]:
# 13) POST-RUN AUDIT — Control selection statistics (levels + criteria)

# 13.1) Inputs: uses existing notebook variables if present
if "CONTROL_LOG" not in globals():
    raise ValueError("CONTROL_LOG is not defined in this notebook.")
if "CASES_DB_OUT" not in globals():
    raise ValueError("CASES_DB_OUT is not defined in this notebook.")

if not os.path.exists(CONTROL_LOG):
    raise FileNotFoundError(f"CONTROL_LOG not found: {CONTROL_LOG}")
if not os.path.exists(CASES_DB_OUT):
    raise FileNotFoundError(f"CASES_DB_OUT not found: {CASES_DB_OUT}")

log = pd.read_csv(CONTROL_LOG, dtype=str).fillna("")
cases = pd.read_csv(CASES_DB_OUT, dtype=str).fillna("")
cases.columns = [c.strip() for c in cases.columns]
cases["code"] = cases["code"].astype(str).str.strip()

# Keep only successfully downloaded controls
dl = log[log["status"].str.lower().eq("downloaded")].copy()
if dl.empty:
    print("No downloaded controls found in the log yet.")
    raise SystemExit

# Join case metadata for criteria auditing
cases_small = cases[["code", "subject", "subject_norm", "article_type", "article_type_norm", "journal", "publisher", "year"]].copy()
cases_small["code"] = cases_small["code"].astype(str).str.strip()
dl["case_code"] = dl["case_code"].astype(str).str.strip()
df = dl.merge(cases_small, left_on="case_code", right_on="code", how="left", suffixes=("", "_case"))

# 13.2) Helpers
def is_blank(x):
    return (x is None) or (str(x).strip() == "") or (str(x).strip().lower() in {"nan", "none"})

def norm_text(x):
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x).strip().lower())

ARTICLETYPE_TO_OPENALEX = {
    "journal article": "journal-article",
    "journal-article": "journal-article",
    "article": "journal-article",
    "research article": "journal-article",
    "review": "review",
    "review article": "review",
    "proceedings article": "proceedings-article",
    "conference paper": "proceedings-article",
    "conference article": "proceedings-article",
    "proceedings-article": "proceedings-article",
    "book chapter": "book-chapter",
    "book-chapter": "book-chapter",
    "preprint": "preprint",
    "dataset": "dataset",
    "editorial": "editorial",
    "letter": "letter",
    "retraction": "retraction",
    "correction": "correction",
    "erratum": "correction",
}
CANON_OPENALEX_TYPES = {
    "journal-article","proceedings-article","book-chapter","preprint","review",
    "dataset","editorial","letter","retraction","correction"
}

def case_article_type_target(row) -> str:
    raw = row.get("article_type_norm", "") or row.get("article_type", "")
    s = norm_text(raw)
    if not s:
        return ""
    return ARTICLETYPE_TO_OPENALEX.get(s, s)

def article_type_matches(case_target: str, work_type: str) -> bool:
    case_target = norm_text(case_target)
    work_type = norm_text(work_type)
    if not case_target:
        return True
    if not work_type:
        return False
    if case_target in CANON_OPENALEX_TYPES:
        return work_type == case_target
    return (case_target in work_type) or (work_type in case_target)

def rate(series_bool):
    if len(series_bool) == 0:
        return np.nan
    return round(series_bool.mean() * 100, 2)

# 13.3) Derived flags
df["level"] = df["match_level"].astype(str).str.strip()

ENFORCED_LEVELS = {"L1", "L1b", "L1c", "L2"}  # only these enforce country/type/subject
df["is_enforced_level"] = df["level"].isin(ENFORCED_LEVELS)
df["is_L4"] = df["level"].eq("L4")
df["is_L3"] = df["level"].eq("L3")

# Subject availability (from cases DB)
df["case_has_subject"] = ~df["subject"].apply(is_blank)

# Country availability + match
df["case_has_country"] = ~df["case_country_iso2"].apply(is_blank)
df["ctrl_has_country"] = ~df["control_first_author_country_iso2"].apply(is_blank)
df["country_match"] = (
    df["case_has_country"] & df["ctrl_has_country"] &
    (df["case_country_iso2"].str.upper() == df["control_first_author_country_iso2"].str.upper())
)

# Article type availability + match
df["case_type_target"] = df.apply(case_article_type_target, axis=1)
df["case_has_type"] = ~df["case_type_target"].apply(is_blank)
df["ctrl_has_type"] = ~df["control_type"].apply(is_blank)
df["type_match"] = df.apply(lambda r: article_type_matches(r["case_type_target"], r["control_type"]), axis=1)

# Enforcement logic
df["country_enforced"] = df["is_enforced_level"] & df["case_has_country"]
df["type_enforced"] = df["is_enforced_level"] & df["case_has_type"]
df["subject_required_for_level"] = df["is_enforced_level"]

# For enforced criteria, matches are expected(or else it would not have been accepted)
df["country_passed_when_enforced"] = np.where(df["country_enforced"], df["country_match"], np.nan)
df["type_passed_when_enforced"] = np.where(df["type_enforced"], df["type_match"], np.nan)

# ---------- TABLE 1: Level distribution ----------
t1 = (
    df.groupby("level")
      .size()
      .rename("n")
      .reset_index()
      .sort_values("n", ascending=False)
)
t1["pct"] = (t1["n"] / t1["n"].sum() * 100).round(2)

# ---------- TABLE 2: Overall criteria availability + match ----------
overall = {
    "Downloaded controls (n)": len(df),

    "Cases with subject available (%)": rate(df["case_has_subject"]),
    "Cases with country available (%)": rate(df["case_has_country"]),
    "Cases with article type available (%)": rate(df["case_has_type"]),

    "Controls with OpenAlex first-author country available (%)": rate(df["ctrl_has_country"]),
    "Controls with OpenAlex type available (%)": rate(df["ctrl_has_type"]),

    "Country match where BOTH countries available (%)": rate(df[df["case_has_country"] & df["ctrl_has_country"]]["country_match"]),
    "Type match where case type available (%)": rate(df[df["case_has_type"]]["type_match"]),

    "Country passed when enforced (should be 100%) (%)": rate(df[df["country_enforced"]]["country_match"]),
    "Type passed when enforced (should be 100%) (%)": rate(df[df["type_enforced"]]["type_match"]),
}
t2 = pd.DataFrame([overall]).T.reset_index()
t2.columns = ["Metric", "Value"]

# ---------- TABLE 3: Criteria match rates by level ----------
def summarise_by_level(g):
    out = {
        "n": len(g),
        "case_has_subject_%": rate(g["case_has_subject"]),
        "case_has_country_%": rate(g["case_has_country"]),
        "ctrl_has_country_%": rate(g["ctrl_has_country"]),
        "country_match_if_both_%": rate(g[g["case_has_country"] & g["ctrl_has_country"]]["country_match"])
                               if len(g[g["case_has_country"] & g["ctrl_has_country"]]) else np.nan,
        "country_enforced_%": rate(g["country_enforced"]),
        "country_passed_when_enforced_%": rate(g[g["country_enforced"]]["country_match"])
                                        if len(g[g["country_enforced"]]) else np.nan,

        "case_has_type_%": rate(g["case_has_type"]),
        "ctrl_has_type_%": rate(g["ctrl_has_type"]),
        "type_match_if_case_type_%": rate(g[g["case_has_type"]]["type_match"])
                                   if len(g[g["case_has_type"]]) else np.nan,
        "type_enforced_%": rate(g["type_enforced"]),
        "type_passed_when_enforced_%": rate(g[g["type_enforced"]]["type_match"])
                                     if len(g[g["type_enforced"]]) else np.nan,
    }
    return pd.Series(out)

t3 = df.groupby("level").apply(summarise_by_level).reset_index().sort_values("n", ascending=False)

# ---------- TABLE 4: L4 diagnostics (indicative) ----------
# Note: indicative because we only observe the accepted control, not all rejected candidates.
l4 = df[df["is_L4"]].copy()
if not l4.empty:
    t4 = pd.DataFrame([{
        "L4 controls (n)": len(l4),
        "Cases missing subject (%)": rate(~l4["case_has_subject"]),
        "Controls missing OpenAlex first-author country (%)": rate(~l4["ctrl_has_country"]),
        "Controls missing OpenAlex type (%)": rate(~l4["ctrl_has_type"]),
    }]).T.reset_index()
    t4.columns = ["Metric", "Value"]
else:
    t4 = pd.DataFrame([["L4 controls (n)", 0]], columns=["Metric", "Value"])

# ---------- Print tables ----------
print("\n=== TABLE 1: Downloaded controls by match level ===")
display(t1)

print("\n=== TABLE 2: Overall criteria availability + match rates ===")
display(t2)

print("\n=== TABLE 3: Criteria rates by match level ===")
display(t3)

print("\n=== TABLE 4: L4 diagnostics (indicative) ===")
display(t4)

# Optional: Save audit outputs to CSV for appendix / reproducibility
AUDIT_DIR = os.path.join(os.path.dirname(CONTROL_LOG), "audit_outputs")
os.makedirs(AUDIT_DIR, exist_ok=True)

t1.to_csv(os.path.join(AUDIT_DIR, "audit_table_levels.csv"), index=False)
t2.to_csv(os.path.join(AUDIT_DIR, "audit_table_overall_metrics.csv"), index=False)
t3.to_csv(os.path.join(AUDIT_DIR, "audit_table_metrics_by_level.csv"), index=False)
t4.to_csv(os.path.join(AUDIT_DIR, "audit_table_L4_diagnostics.csv"), index=False)

print(f"\nAudit tables saved to: {AUDIT_DIR}")